# 12.2: Merancang Alur Kerja sebagai Graph

**Empat komponen yang akan kita pakai**

| Komponen | Artinya |
|---|---|
| **State** | Catatan bersama, dibaca dan ditulis seluruh alur |
| **Nodes** | Satu node, satu pekerjaan |
| **Edges** | Penghubung yang menentukan urutan |
| **Conditional Edges** | Penghubung yang memilih tujuan berdasarkan kondisi |

Hanya komponen keempat yang benar-benar baru. Tiga yang pertama adalah cara
formal untuk menuliskan hal yang sudah Anda lakukan sejak Sesi 11.

## 0. Pasang LangGraph

Satu baris, dijalankan sekali. Ringan, karena dependency utamanya sudah
terpasang sejak Sesi 11.

`grandalf` dipakai untuk menggambar graph di dalam notebook tanpa memanggil
layanan luar. Tanpa itu, `draw_ascii()` akan menolak jalan.

In [ ]:
# Jalankan sekali saja, lalu restart kernel apabila diminta.
%pip install -q langgraph grandalf

In [ ]:
import os
import sys
import uuid

sys.path.insert(0, "..")

from typing_extensions import TypedDict

from dotenv import load_dotenv
from google import genai

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import interrupt, Command

from rag_sesi11 import cari

load_dotenv()
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-2.5-flash"


def tanya_model(prompt):
    resp = client.models.generate_content(model=MODEL, contents=prompt)
    return resp.text.strip()


print("siap")

## Live Coding : Membuat Graph

Kita bangun graph paling kecil yang tetap ada gunanya: cari dulu, lalu putuskan
apakah hasilnya layak dijawab atau tidak.

Perhatikan urutan kerjanya. Kita mendefinisikan **state**, lalu **node** sebagai
fungsi Python biasa, lalu **edge** untuk menyambungkannya, lalu `compile()`.

Satu aturan yang membingungkan di awal: sebuah node **mengembalikan perubahan**,
bukan mengubah state secara langsung. Yang dikembalikan akan digabungkan ke
state oleh LangGraph.

In [ ]:
class State(TypedDict):
    pertanyaan: str
    potongan: list
    hasil: str


def cari_node(state: State):
    potongan = cari(state["pertanyaan"], k=3)
    return {"potongan": potongan}


def jawab_node(state: State):
    blok = "\n\n".join(p["teks"] for p in state["potongan"])
    teks = tanya_model(
        f"Jawab singkat berdasarkan konteks.\n\n{blok}\n\n"
        f"PERTANYAAN: {state['pertanyaan']}"
    )
    return {"hasil": teks}


def menyerah_node(state: State):
    return {"hasil": "Maaf, informasinya tidak ada di knowledge base."}

Sekarang komponen yang baru. `ketemu_tidak` bukan sebuah node. Ia adalah fungsi
yang **membaca state dan mengembalikan nama tujuan**. LangGraph memanggilnya di
persimpangan, lalu mengikuti nama yang dikembalikan.

In [ ]:
def ketemu_tidak(state: State):
    tertinggi = max(p["skor"] for p in state["potongan"])
    # TODO 1: kembalikan "ya" apabila skor tertinggi di atas 0.30,
    # dan "tidak" apabila tidak.
    return None


builder = StateGraph(State)
builder.add_node("cari", cari_node)
builder.add_node("jawab", jawab_node)
builder.add_node("menyerah", menyerah_node)

builder.add_edge(START, "cari")
builder.add_conditional_edges(
    "cari", ketemu_tidak,
    {"ya": "jawab", "tidak": "menyerah"})
builder.add_edge("jawab", END)
builder.add_edge("menyerah", END)

graph = builder.compile()
print("graph terbentuk")

Cetak graphnya.

In [ ]:
print(graph.get_graph().draw_ascii())

In [ ]:
hasil = graph.invoke({"pertanyaan": "Berapa lama masa tunggu klaim rawat inap?"})
print(hasil["hasil"])

print()
hasil = graph.invoke({"pertanyaan": "Berapa harga tiket konser di Jakarta?"})
print(hasil["hasil"])

## 2. Live Coding 2: agen yang menilai hasilnya sendiri

Sebuah graph boleh **berputar**. Ia bisa kembali ke node sebelumnya untuk
mencoba lagi, yang tidak mungkin dilakukan chain linear.

Kita bangun agen sederhana: cari, nilai apakah cukup, kalau belum cukup cari
lagi dengan k yang lebih besar, kalau sudah cukup tulis jawabannya.

Kita batasi tiga putaran. Tanpa batas, sebuah graph siklik bisa berputar selama
mana pun, dan itu berarti tagihan API yang tidak Anda kendalikan.

In [ ]:
class StateUlang(TypedDict):
    pertanyaan: str
    potongan: list
    putaran: int
    cukup: bool
    hasil: str


def cari_ulang(state: StateUlang):
    putaran = state.get("putaran", 0) + 1
    k = 2 * putaran          # 2, lalu 4, lalu 6
    return {"potongan": cari(state["pertanyaan"], k=k), "putaran": putaran}


def nilai(state: StateUlang):
    tertinggi = max(p["skor"] for p in state["potongan"])
    return {"cukup": tertinggi > 0.35}


def tulis(state: StateUlang):
    blok = "\n\n".join(p["teks"] for p in state["potongan"])
    teks = tanya_model(
        f"Jawab singkat berdasarkan konteks.\n\n{blok}\n\n"
        f"PERTANYAAN: {state['pertanyaan']}"
    )
    return {"hasil": teks}

In [ ]:
def cukup_belum(state: StateUlang):
    # TODO 2: kembalikan "tulis" apabila state["cukup"] benar, ATAU apabila
    # putaran sudah mencapai 3. Selain itu kembalikan "cari".
    return None


b = StateGraph(StateUlang)
b.add_node("cari", cari_ulang)
b.add_node("nilai", nilai)
b.add_node("tulis", tulis)

b.add_edge(START, "cari")
b.add_edge("cari", "nilai")
b.add_conditional_edges("nilai", cukup_belum,
                        {"cari": "cari", "tulis": "tulis"})
b.add_edge("tulis", END)

agen = b.compile()
print(agen.get_graph().draw_ascii())

In [ ]:
hasil = agen.invoke({"pertanyaan": "Apa saja langkah prosedur banding?",
                     "putaran": 0})
print("putaran terpakai:", hasil["putaran"])
print(hasil["hasil"])

Inilah yang membuat sebuah alur pantas disebut agen: ia menilai hasil kerjanya
sendiri, lalu memutuskan apakah perlu mengulang. Bukan karena kita menyuruhnya
mengulang, melainkan karena kondisinya belum terpenuhi.

## 3. Live Coding : Multi - agent System

Sebuah agen bukan chatbot. Agen adalah node yang punya satu pekerjaan dan satu
prompt.

Kita buat dua. **Researcher** mengumpulkan bahan. **Writer** menyusun jawaban
akhir dari bahan itu. Keduanya hidup dalam satu graph, dan output yang satu
menjadi input yang lain lewat state.

In [ ]:
class StateTim(TypedDict):
    topik: str
    bahan: str
    artikel: str


def researcher(state: StateTim):
    potongan = cari(state["topik"], k=5)
    bahan = "\n".join(f"- {p['teks'][:200]}" for p in potongan)
    return {"bahan": bahan}


def writer(state: StateTim):
    # TODO 3: minta model menyusun penjelasan rapi maksimal tiga paragraph
    # untuk nasabah, berdasarkan state["bahan"] dan state["topik"].
    # Kembalikan {"artikel": teks}.
    return None


t = StateGraph(StateTim)
t.add_node("researcher", researcher)
t.add_node("writer", writer)
t.add_edge(START, "researcher")
t.add_edge("researcher", "writer")
t.add_edge("writer", END)

tim = t.compile()
print(tim.get_graph().draw_ascii())

In [ ]:
hasil = tim.invoke({"topik": "prosedur banding klaim"})
print(hasil["artikel"])

Researcher tidak tahu apa pun tentang Writer. Ia hanya menulis ke `bahan`.
Writer tidak tahu dari mana `bahan` datang. Keduanya berkomunikasi lewat state,
bukan lewat pemanggilan langsung.

Itulah alasan pola ini bisa tumbuh. Menambah agen ketiga berarti menambah satu
node dan satu edge, bukan menulis ulang dua agen yang sudah ada.

## Live Coding :  Human in the loop

Sebagian tindakan tidak boleh dijalankan AI tanpa persetujuan manusia. Mengirim
email, memindahkan uang, menutup tiket.

`interrupt()` menghentikan graph di tengah jalan dan menunggu. Anda melanjutkannya
dengan `Command(resume=...)`.

Dua hal yang wajib:

1. graph harus di-`compile` dengan sebuah **checkpointer**, karena state harus
   disimpan selama ia menunggu. `InMemorySaver` cukup untuk belajar.
2. Setiap jalannya graph butuh **`thread_id`**, dan `thread_id` yang sama harus
   dipakai saat melanjutkan.

In [ ]:
class StateSetuju(TypedDict):
    topik: str
    draft: str
    disetujui: bool
    akhir: str


def susun_draft(state: StateSetuju):
    teks = tanya_model(f"Tulis satu paragraph email singkat tentang {state['topik']}.")
    return {"draft": teks}


def konfirmasi(state: StateSetuju):
    # TODO 4: panggil interrupt() dengan {"draft": state["draft"]}, simpan
    # hasilnya, lalu kembalikan {"disetujui": ...} bernilai True apabila
    # jawabannya "ya".
    return None


def kirim(state: StateSetuju):
    if state["disetujui"]:
        return {"akhir": "Email dikirim."}
    return {"akhir": "Dibatalkan, email tidak dikirim."}


s = StateGraph(StateSetuju)
s.add_node("susun_draft", susun_draft)
s.add_node("konfirmasi", konfirmasi)
s.add_node("kirim", kirim)
s.add_edge(START, "susun_draft")
s.add_edge("susun_draft", "konfirmasi")
s.add_edge("konfirmasi", "kirim")
s.add_edge("kirim", END)

alur = s.compile(checkpointer=InMemorySaver())
print("alur siap")

In [ ]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}

hasil = alur.invoke({"topik": "keterlambatan proses klaim"}, config=config)

print("ALUR BERHENTI DI SINI.")
print("Draft yang menunggu persetujuan:")
print()
print(hasil["__interrupt__"][0].value["draft"])

In [ ]:
# Lanjutkan dengan jawaban manusia. Ganti "ya" menjadi "tidak" lalu jalankan
# ulang sel sebelumnya untuk melihat cabang yang lain.
akhir = alur.invoke(Command(resume="ya"), config=config)
print(akhir["akhir"])

### Satu jebakan yang wajib Anda tahu

Saat alur dilanjutkan, node yang berisi `interrupt()` dijalankan **ulang dari
awal**. Semua kode di atas baris `interrupt()` berjalan untuk kedua kalinya.

Kalau di atas `interrupt()` ada pemanggilan API, ia terpanggil dua kali dan
Anda membayar dua kali. Kalau ada penulisan ke database, ia tertulis dua kali.

Aturannya sederhana: letakkan hal yang berdampak **setelah** `interrupt()`,
bukan sebelumnya.

Buktikan sendiri.

In [ ]:
class StateBukti(TypedDict):
    nilai: str


def node_bukti(state: StateBukti):
    # TODO 5: letakkan satu print() di atas baris interrupt, lalu panggil
    # interrupt({"tanya": "lanjut?"}). Jalankan sel ini dan hitung berapa
    # kali print itu muncul.
    jawaban = interrupt({"tanya": "lanjut?"})
    return {"nilai": jawaban}


bb = StateGraph(StateBukti)
bb.add_node("bukti", node_bukti)
bb.add_edge(START, "bukti")
bb.add_edge("bukti", END)
bukti = bb.compile(checkpointer=InMemorySaver())

cfg = {"configurable": {"thread_id": str(uuid.uuid4())}}
bukti.invoke({"nilai": ""}, config=cfg)
print("--- melanjutkan ---")
bukti.invoke(Command(resume="ya"), config=cfg)

## Rangkuman

Empat komponen, dan apa yang masing-masing menyelesaikan:

| Komponen | Menyelesaikan |
|---|---|
| State | Langkah keempat masih tahu temuan langkah pertama |
| Nodes | Instruksi besar terpecah menjadi pekerjaan kecil |
| Edges | Urutan yang tetap, di bagian yang memang tetap |
| Conditional Edges | Tempat untuk "jika" yang hilang di 12.1 |

Yang masih kurang satu: sistem ini tetap tidak bisa menghitung, dan tidak bisa
melihat data di luar knowledge base. Itu 12.3.